<a href="https://colab.research.google.com/github/Mario-Cattaneo/SemesterProject/blob/main/Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Load Disnet splitting repository

In [1]:
# Clone the repository
!git clone https://github.com/ricsamikwa/DiSNet.git
%cd DiSNet

# Install any required packages (check requirements.txt if present in the repo)
# !pip install -r requirements.txt

fatal: destination path 'DiSNet' already exists and is not an empty directory.
/content/DiSNet


Modify deprecated components

In [2]:
# add better documentation
# 1. Patch model_vgg16.py to fix the super() reload bug
with open('models/model_vgg16.py', 'r') as file:
    content = file.read()
content = content.replace("super(VGG16, self).__init__()", "super().__init__()")
with open('models/model_vgg16.py', 'w') as file:
    file.write(content)
print("✓ Patched model_vgg16.py to prevent reload bugs.")

# 2. Patch model_convert.py to fix paths and torchvision imports
with open('main/model_convert.py', 'r') as file:
    content = file.read()
old_line = "Vgg_pretrined = torch.hub.load('pytorch/vision:v0.9.0', 'vgg16_bn', pretrained=True)"
new_line = "import torchvision.models as models\nVgg_pretrined = models.vgg16_bn(pretrained=True)"
content = content.replace(old_line, new_line)
old_path = "/home/eric/.cache/torch/hub/checkpoints/vgg16_bn-6c64b313.pth"
new_path = "/root/.cache/torch/hub/checkpoints/vgg16_bn-6c64b313.pth"
content = content.replace(old_path, new_path)
with open('main/model_convert.py', 'w') as file:
    file.write(content)
print("✓ Patched main/model_convert.py for Colab compatibility.")

✓ Patched model_vgg16.py to prevent reload bugs.
✓ Patched main/model_convert.py for Colab compatibility.


Specify devices and splitting configuration

In [3]:
# add better documentation
%%writefile configurations.py
import numpy as np

SERVER_ADDR = '192.168.1.38'
SERVER_PORT = 43000

N = 50000 # Data length

model_cfg = {
    'VGG' : [
        ('C', 3, 32, 3, 32*32*32, 32*32*32*3*3*3), ('M', 32, 32, 2, 32*16*16, 0),
        ('C', 32, 64, 3, 64*16*16, 64*16*16*3*3*32), ('M', 64, 64, 2, 64*8*8, 0),
        ('C', 64, 64, 3, 64*8*8, 64*8*8*3*3*64),
        ('D', 8*8*64, 128, 1, 64, 128*8*8*64),
        ('C', 64, 64, 3, 64*8*8, 64*8*8*3*3*64),
        ('D', 8*8*64, 128, 1, 64, 128*8*8*64),
        ('D', 128, 10, 1, 10, 128*10)
    ]
}

model_name = 'VGG'
model_size = 1.28
split_layer = [2, 3, 2]
model_len = 7
DEVICE_PACE_RATE = 10
rate = 10

max_par_partitions = [6, 5, 4, 3]

trans_powers = [1, 0.8, 1.1]
comp_powers = [3.8, 2.2, 1.2]
device_power_groups = [3, 7, 10]

pos_max_par_partitions = [4, 9, 12, 18]
layer_range = np.array([[0, 4], [4, 9], [9, 14], [14, 18]])

R = 100
LR = 0.01
B = 100

K = 10

iteration = {'192.168.1.33': 5, '192.168.1.41': 5, '192.168.1.40': 10, '192.168.1.42': 5, '192.168.1.43': 5}
random = True
random_seed = 0

Overwriting configurations.py


Perform splitting and extract the network and micro-splits

In [4]:
# to extract the network and micro-splits into a compatible/easier
# to work with format for later and also print the network

Coding scheme interface

In [ ]:
# move the interface into a dedicated cell and also add the training into the interface
# for models which don't train this can then simply be set in an appropriate way.
# For evaluation we will have to be able to pass in the bandwidth and latency we want,
# such that the model decides how much to compress to achieve the latency under the given bandwidth
# It can be a deign choice whether the a instance is supposed to be trained only once and instantiated for a specific latency and bandwidth,
# or if can be retrained/(online learn) for new data and or for new latency, bandwidt, as long as for the evaluation we can specify the
# latency and bandwidth.

Basic quantization

In [ ]:
# define here

Basic pruning

In [ ]:
# define here

DAG-VDDBID

In [7]:
# define here

Evaluation

In [ ]:
# for every model iterate over a latency range for a fixed bandwidth, and train the model at every latency level
# and then print the accuracy achieved during testing